In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Import PyTorch packages
import torch
import torch.nn as nn

In [ ]:
# Check the version of PyTorch
print(torch.__version__)

In [ ]:
if not torch.cuda.is_available():
    raise SystemError(
        "No GPU found!\n"
        "To enable GPU, click on top menu: Runtime → Change runtime type → Hardware Accelerator → T4 GPU → Save"
    )

device = torch.device('cuda')
print(f'Using device: {torch.cuda.get_device_name(0)}')

In [ ]:
# Access to Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Load the SELECTED (Top 30) Feature Dataset
* Results of ML3-1 and ML3-2

In [ ]:
FeatureSelected = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/SavedFiles/FeatureSelected.csv', header=None)
FeatureSelected = FeatureSelected.T
FeatureSelected.shape

In [ ]:
# Standardize feature values
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

FeatureSelected_std = StandardScaler().fit_transform(FeatureSelected)
FeatureSelected_std = torch.tensor(FeatureSelected_std, dtype=torch.float32).to(device) #converting the resulting NumPy array into a PyTorch tensor, PyTorch expects 32-bit floats by default, moves tensor to GPU
FeatureSelected_std.shape

## Split Training & Test Data
- Use 'train_test_split' function
- It randomly samples the training and testing data according to the designated ratio.

In [ ]:
# Number of data for each condition: 180
NoOfData   = int(FeatureSelected_std.shape[0]/2)

NormalSet   = FeatureSelected_std[:NoOfData , :]
AbnormalSet = FeatureSelected_std[NoOfData: , :]

NormalSet.shape, AbnormalSet.shape

In [ ]:
from sklearn.model_selection import train_test_split

# Designate test data ratio
TestData_Ratio = 0.2

# Move back to CPU for sklearn compatibility, split, then back to GPU
TrainData_Nor, TestData_Nor = train_test_split(NormalSet.to('cpu').numpy()  , test_size=TestData_Ratio, random_state=777)
TrainData_Abn, TestData_Abn = train_test_split(AbnormalSet.to('cpu').numpy(), test_size=TestData_Ratio, random_state=777)

# Convert back to PyTorch tensors on GPU
TrainData_Nor = torch.tensor(TrainData_Nor, dtype=torch.float32).to(device)
TestData_Nor  = torch.tensor(TestData_Nor , dtype=torch.float32).to(device)
TrainData_Abn = torch.tensor(TrainData_Abn, dtype=torch.float32).to(device)
TestData_Abn  = torch.tensor(TestData_Abn , dtype=torch.float32).to(device)

print(TrainData_Nor.shape, TestData_Nor.shape)
print(TrainData_Abn.shape, TestData_Abn.shape)

## Data Labeling
- Use 'np.zeros' and 'np.ones'
- '[1,0]' refers to 'Normal' and '[1,0]' refers to 'Abnormal' in this tutorial

In [ ]:
# Normal = 0, Abnormal = 1
TrainLabel_Nor = torch.zeros(TrainData_Nor.shape[0], dtype=torch.long).to(device)  # [0,0,0,...] Normal
TrainLabel_Abn = torch.ones( TrainData_Abn.shape[0], dtype=torch.long).to(device)  # [1,1,1,...] Abnormal
TestLabel_Nor  = torch.zeros(TestData_Nor.shape[0],  dtype=torch.long).to(device)  # [0,0,0,...] Normal
TestLabel_Abn  = torch.ones( TestData_Abn.shape[0],  dtype=torch.long).to(device)  # [1,1,1,...] Abnormal

print(TrainLabel_Nor.shape, TestLabel_Nor.shape)
print(TrainLabel_Abn.shape, TestLabel_Abn.shape)

In [ ]:
TestLabel_Nor

## Data and Label Preparation

In [ ]:
TrainData  = torch.cat([TrainData_Nor,  TrainData_Abn ], dim=0)
TestData   = torch.cat([TestData_Nor,   TestData_Abn  ], dim=0)
TrainLabel = torch.cat([TrainLabel_Nor, TrainLabel_Abn], dim=0)
TestLabel  = torch.cat([TestLabel_Nor,  TestLabel_Abn ], dim=0)

print(TrainData.shape,  TestData.shape)
print(TrainLabel.shape, TestLabel.shape)

## Setting hyperparameters for training ANN(Artificial Neural Network)

In [ ]:
learningRate  = 0.001
noOfNeuron    = 16
Epoch         = 200

## Designing an ANN architecture

In [ ]:
class ANN_model(nn.Module):
    def __init__(self, input_size):
        super(ANN_model, self).__init__()
        self.hidden1 = nn.Linear(input_size, noOfNeuron)
        self.hidden2 = nn.Linear(noOfNeuron, noOfNeuron)
        self.output  = nn.Linear(noOfNeuron, 2)
        self.relu    = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        x = self.output(x)
        return x

AnnModel  = ANN_model(TrainData.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(AnnModel.parameters(), lr=learningRate)

In [ ]:
# Print model summary
print(AnnModel)
total_params     = sum(p.numel() for p in AnnModel.parameters())
trainable_params = sum(p.numel() for p in AnnModel.parameters() if p.requires_grad)
print(f'\nTotal parameters:     {total_params}')
print(f'Trainable parameters: {trainable_params}')
for name, param in AnnModel.named_parameters():
    print(f'{name}: {param.shape}')

## ANN Model Training

In [ ]:
torch.manual_seed(777)
AnnModel.train()
TrainingHistory = {'loss': [], 'accuracy': []}

idx        = torch.randperm(TrainData.shape[0])   #This prevents the model from learning the order of the data
TrainData  = TrainData[idx]
TrainLabel = TrainLabel[idx]

for epoch in range(Epoch):
    optimizer.zero_grad()  #Clears gradients from the previous epoch. PyTorch accumulates gradients by default so this must be done at the start of every epoch.
    outputs  = AnnModel(TrainData)  #feeds all training data through the network and gets raw predictions. Internally calls forward().
    loss     = criterion(outputs, TrainLabel)  #measures how wrong the predictions are using CrossEntropyLoss.
    loss.backward()  #computes gradients of the loss with respect to every weight in the network via backpropagation.
    optimizer.step() #Adam optimizer uses the computed gradients to update all weights:

    predicted = torch.argmax(outputs, dim=1)  #Converts raw output scores to predicted class (0 or 1) by picking the index of the highest score.
    accuracy  = (predicted == TrainLabel).float().mean().item()

    TrainingHistory['loss'].append(loss.item())
    TrainingHistory['accuracy'].append(accuracy)

    if (epoch+1) % 10 == 0:  #Prints progress every 10 epochs.
        print(f'Epoch [{epoch+1}/{Epoch}] Loss: {loss.item():.4f}  Accuracy: {accuracy:.4f}')

In [ ]:
# Evaluation result for test data (not trained)
AnnModel.eval()
with torch.no_grad():                     #Disables gradients, gradients are only needed for the backward pass to update weights during learning, not needed during eval and test
    outputs   = AnnModel(TestData)
    loss      = criterion(outputs, TestLabel)
    predicted = torch.argmax(outputs, dim=1)
    accuracy  = (predicted == TestLabel).float().mean().item()

print(f'Loss: {loss.item():.4f}  Accuracy: {accuracy:.4f}')
print('The closer the Loss is to 0 and the closer the accuracy is to 1 (100%), the better.')

In [ ]:
# Check the training process (Loss, Accuracy)

fig, loss_ax = plt.subplots(figsize=(8,6))
acc_ax = loss_ax.twinx()

loss_ax.plot(TrainingHistory['loss'],     label='train loss', c='tab:red')
loss_ax.set_xlabel('epoch', fontsize=15)
loss_ax.set_ylabel('loss',  fontsize=15)
loss_ax.legend(loc='upper left', fontsize=15)

acc_ax.plot(TrainingHistory['accuracy'], label='train acc', c='tab:blue')
acc_ax.set_ylabel('accuracy', fontsize=15)
acc_ax.legend(loc='lower left', fontsize=15)

plt.show()

### [Tip] Defining a callback function to check the training progress at desired epochs

In [ ]:
# Define the callback function
EpochForPrint = 20

class CheckProcess:
    def on_epoch_end(self, epoch, logs=None):
        if epoch % EpochForPrint == 0:
            print("{} Epochs Train Acc. : {:.2f}%  ".format(epoch, logs["accuracy"]*100))

In [ ]:
# Fresh model instance
AnnModel_2  = ANN_model(TrainData.shape[1]).to(device)
optimizer_2 = torch.optim.Adam(AnnModel_2.parameters(), lr=learningRate)

callback = CheckProcess()

AnnModel_2.train()
TrainingHistory_2 = {'loss': [], 'accuracy': []}

for epoch in range(Epoch):
    optimizer_2.zero_grad()
    outputs  = AnnModel_2(TrainData)
    loss     = criterion(outputs, TrainLabel)
    loss.backward()
    optimizer_2.step()

    predicted = torch.argmax(outputs, dim=1)
    accuracy  = (predicted == TrainLabel).float().mean().item()

    TrainingHistory_2['loss'].append(loss.item())
    TrainingHistory_2['accuracy'].append(accuracy)

    # Manually trigger callback
    callback.on_epoch_end(epoch, logs={"accuracy": accuracy})

print('Final Train Accuracy : {:.2f}%'.format(TrainingHistory_2['accuracy'][-1]*100))

#Save ML model (ANN) as a file

In [ ]:
import os

# Create directory if it doesn't exist
os.makedirs('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models', exist_ok=True)

# Then save
torch.save(AnnModel.state_dict(), '/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/ANN_model.pth')

#Load the saved ML model (ANN) and test

In [ ]:
# Load the saved model (ANN_model() + load_state_dict())
LoadedModel = ANN_model(TrainData.shape[1]).to(device)
LoadedModel.load_state_dict(torch.load('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/ANN_model.pth'))

# Evaluate, must rebuild architecture manually first in PyTorch
LoadedModel.eval()
with torch.no_grad():                     #Disables gradients, gradients are only needed for the backward pass to update weights during learning, not needed during eval and test
    outputs   = LoadedModel(TestData)
    loss      = criterion(outputs, TestLabel)
    predicted = torch.argmax(outputs, dim=1)
    accuracy  = (predicted == TestLabel).float().mean().item()

print('[Performance of ANN model] \n')
print('Accuracy : {:.2f}%'.format(accuracy*100))

In [ ]:
# Predicted result
LoadedModel.eval()
with torch.no_grad():
    Predicted = LoadedModel(TestData)
    Predicted = torch.softmax(Predicted, dim=1)

pd.DataFrame(Predicted.cpu().numpy())

#Why softmax is applied here: Softmax was removed from forward() during training. nn.CrossEntropyLoss already applies softmax internally as part of its loss calculation.
#Now during inference it is added back manually to convert raw scores into actual probabilities that sum to 1: